In [27]:
import requests
import pandas as pd
import joblib
import numpy as np
import re

BASE_URL = "https://sdp-prem-prod.premier-league-prod.pulselive.com/api/v2/matches"
competition_id = 8
season_id = 2025

output_all = "../data/raw/football_data/premier_league_2025_26_fixtures.csv"
output_upcoming = "../data/processed/premier_league_2025_26_upcoming_prediction_template.csv"
output_annotated = "../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv"

feature_path = "../models/weekly/homewin/feature_list_20260327T205139Z.pkl"

def sanitize_team(s):
    return re.sub(r"\W+", "_", str(s).strip())

def add_rolling_features(df, n_matches=5):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Kickoff"], errors="coerce")
    df = df.sort_values(["Date", "MatchId"]).reset_index(drop=True)
   
    df["FTHG"] = pd.to_numeric(df["FTHG"], errors="coerce")
    df["FTAG"] = pd.to_numeric(df["FTAG"], errors="coerce")
  
    df["HomeTeam_mean_FTHG"] = (
        df.groupby("HomeTeam")["FTHG"]
        .apply(lambda x: x.shift(1).expanding().mean())
        .reset_index(level=0, drop=True)
    )
    df["AwayTeam_mean_FTAG"] = (
        df.groupby("AwayTeam")["FTAG"]
        .apply(lambda x: x.shift(1).expanding().mean())
        .reset_index(level=0, drop=True)
    )
   
    records = []
    for idx, row in df.iterrows():
        past_home = df[
            (df["HomeTeam"] == row["HomeTeam"]) &
            (df["Date"] < row["Date"]) &
            df["FTHG"].notna()
        ].sort_values("Date").tail(n_matches)
        home_gf = past_home["FTHG"].mean() if not past_home.empty else np.nan
        home_ga = past_home["FTAG"].mean() if not past_home.empty else np.nan
        home_pts = past_home.apply(
            lambda r: 3 if r["FTHG"] > r["FTAG"] else (1 if r["FTHG"] == r["FTAG"] else 0), axis=1
        ).sum() if not past_home.empty else np.nan
        
        past_away = df[
            (df["AwayTeam"] == row["AwayTeam"]) &
            (df["Date"] < row["Date"]) &
            df["FTAG"].notna()
        ].sort_values("Date").tail(n_matches)
        away_gf = past_away["FTAG"].mean() if not past_away.empty else np.nan
        away_ga = past_away["FTHG"].mean() if not past_away.empty else np.nan
        away_pts = past_away.apply(
            lambda r: 3 if r["FTAG"] > r["FTHG"] else (1 if r["FTAG"] == r["FTHG"] else 0), axis=1
        ).sum() if not past_away.empty else np.nan
        
        records.append({
            "HomeRecentGF": home_gf,
            "HomeRecentGA": home_ga,
            "HomeRecentPts": home_pts,
            "AwayRecentGF": away_gf,
            "AwayRecentGA": away_ga,
            "AwayRecentPts": away_pts,
        })
    roll_df = pd.DataFrame(records, index=df.index)
    df = pd.concat([df, roll_df], axis=1)
    return df

all_rows = []
for mw in range(1, 39):
    params = {
        "competition": competition_id,
        "season": season_id,
        "matchweek": mw,
        "_limit": 20
    }
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(BASE_URL, params=params, headers=headers)
    js = resp.json()
    for match in js.get("data", []):
        home = match.get("homeTeam", {}).get("name", "")
        away = match.get("awayTeam", {}).get("name", "")
        home_score = match.get("homeTeam", {}).get("score", "")
        away_score = match.get("awayTeam", {}).get("score", "")
        kickoff = match.get("kickoff", "")
        venue = match.get("ground", "")
        matchweek = match.get("matchWeek", "")
        match_id = match.get("matchId", "")
        all_rows.append({
            "MatchWeek": matchweek,
            "HomeTeam": home,
            "AwayTeam": away,
            "FTHG": home_score,
            "FTAG": away_score,
            "Kickoff": kickoff,
            "Venue": venue,
            "MatchId": match_id
        })

df_all = pd.DataFrame(all_rows)
df_all.to_csv(output_all, index=False)

df_all = add_rolling_features(df_all, n_matches=5)

mask_unplayed = (
    (df_all["FTHG"].isna() | (df_all["FTHG"].astype(str).str.strip() == "")) &
    (df_all["FTAG"].isna() | (df_all["FTAG"].astype(str).str.strip() == ""))
)
df_upcoming = df_all[mask_unplayed].copy()

if "Kickoff" in df_upcoming.columns:
    df_upcoming["Date"] = pd.to_datetime(df_upcoming["Kickoff"], errors="coerce").dt.date

feature_list = joblib.load(feature_path)
for col in feature_list:
    if col not in df_upcoming.columns:
        df_upcoming[col] = np.nan

df_model_ready = df_upcoming[feature_list]

extra_cols = ["HomeTeam", "AwayTeam", "Date"]
cols_to_include = [col for col in extra_cols if col in df_upcoming.columns] + list(df_model_ready.columns)
seen = set()
cols_final = []
for c in cols_to_include:
    if c not in seen:
        cols_final.append(c)
        seen.add(c)
df_annotated = df_upcoming[cols_final]

def one_hot_team(df, col_stem, team_col):
    team_cols = [col for col in df.columns if col.startswith(col_stem)]
    for col in team_cols:
        df[col] = 0
    for idx, team in df[team_col].items():
        colname = f"{col_stem}{sanitize_team(team)}"
        if colname in df.columns:
            df.at[idx, colname] = 1

one_hot_team(df_annotated, "AwayTeam_", "AwayTeam")
one_hot_team(df_annotated, "HomeTeam_", "HomeTeam")

df_annotated['Year'] = pd.to_datetime(df_annotated['Date']).dt.year
df_annotated['Month'] = pd.to_datetime(df_annotated['Date']).dt.month
df_annotated['DayOfWeek'] = pd.to_datetime(df_annotated['Date']).dt.dayofweek

df_model_ready.to_csv(output_upcoming, index=False)
df_annotated.to_csv(output_annotated, index=False)

print(f"Model-ready prediction template saved to {output_upcoming}, shape: {df_model_ready.shape}")
print(f"Annotated prediction template (with teams/dates) saved to {output_annotated}, shape: {df_annotated.shape}")

print(df_annotated.filter(like="HomeTeam_").head())
print(df_annotated.filter(like="AwayTeam_").head())
print(df_annotated[['Date', 'Year', 'Month', 'DayOfWeek']].head())

print("\nSample annotated version:")
print(df_annotated.head().T)

Model-ready prediction template saved to ../data/processed/premier_league_2025_26_upcoming_prediction_template.csv, shape: (70, 183)
Annotated prediction template (with teams/dates) saved to ../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv, shape: (70, 186)
     HomeTeam_mean_FTHG  HomeTeam_Arsenal  HomeTeam_Aston_Villa  \
310                   0                 0                     0   
311                   0                 1                     0   
312                   0                 0                     0   
313                   0                 0                     0   
314                   0                 0                     0   

     HomeTeam_Birmingham  HomeTeam_Blackburn  HomeTeam_Blackpool  \
310                    0                   0                   0   
311                    0                   0                   0   
312                    0                   0                   0   
313                    0                  

/var/folders/rj/ybzbgv510dd6rzmm3p9vf0000000gn/T/ipykernel_88004/1551544253.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_upcoming[col] = np.nan
